In [ ]:
!pip install fastapi nest-asyncio uvicorn transformers torch omegaconf

In [ ]:
yaml_config = """
model_path: "unitary/toxic-bert"
"""
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class ToxicityClassification:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.config.model_path)

    def __call__(self, message):
        inputs = self.tokenizer(message, return_tensors="pt")
        with torch.no_grad():
            logits = self.model(**inputs).logits

        probabilities = torch.sigmoid(logits)[0]

        toxic_score = probabilities[0].item()

        if toxic_score > 0.5:
            return "toxic"
        else:
            return "safe"

classifier = ToxicityClassification("./config.yaml")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import threading
import uvicorn

nest_asyncio.apply()

app = FastAPI(title="Hệ thống Kiểm duyệt API")

app.add_middleware(
    CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'],
)

@app.get("/")
def home():
    return {"message": "Chào mừng đến với API chạy mô hình unitary/toxic-bert"}

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post('/predict')
async def predict_toxicity(message: str):
    result = classifier(message)
    return {
        "text": message,
        "is_toxic": True if result == 'toxic' else False,
        "label": result
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server đã chạy thành công tại địa chỉ http://127.0.0.1:8000")

Server đã chạy thành công tại địa chỉ http://127.0.0.1:8000


In [ ]:
import requests
import time
from requests.exceptions import ConnectionError

API_URL = "http://127.0.0.1:8000/predict"

print("TEST API")

max_retries = 10
retry_delay_seconds = 2

def make_request_with_retry(url, params, test_name):
    for i in range(max_retries):
        try:
            response = requests.post(url, params=params)
            print(f"\n{test_name}:")
            print(response.json())
            return
        except ConnectionError as e:
            print(f"Lỗi kết nối ({test_name}, thử lại {i+1}/{max_retries}): {e}")
            time.sleep(retry_delay_seconds)
    print(f"Không thể kết nối đến server sau {max_retries} lần thử cho {test_name}.")

params_1 = {"message": "I love this application, it is very helpful!"}
make_request_with_retry(API_URL, params_1, "Test 1 (Câu lịch sự)")

params_2 = {"message": "You are a stupid and toxic idiot!"}
make_request_with_retry(API_URL, params_2, "Test 2 (Câu độc hại)")

INFO:     Started server process [2875]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


TEST API
Lỗi kết nối (Test 1 (Câu lịch sự), thử lại 1/10): HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /predict?message=I+love+this+application%2C+it+is+very+helpful%21 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d49a9788530>: Failed to establish a new connection: [Errno 111] Connection refused'))
INFO:     127.0.0.1:37210 - "POST /predict?message=I+love+this+application%2C+it+is+very+helpful%21 HTTP/1.1" 200 OK

Test 1 (Câu lịch sự):
{'text': 'I love this application, it is very helpful!', 'is_toxic': False, 'label': 'safe'}
INFO:     127.0.0.1:37226 - "POST /predict?message=You+are+a+stupid+and+toxic+idiot%21 HTTP/1.1" 200 OK

Test 2 (Câu độc hại):
{'text': 'You are a stupid and toxic idiot!', 'is_toxic': True, 'label': 'toxic'}
